<a href="https://colab.research.google.com/github/phuong-duong/coconut-iqa/blob/feat%2Fdisease-classifier/notebooks/disease_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Disease classifier — dự đoán out-of-fold dùng chung

Một classifier bệnh đa lớp (5 thư mục Mendeley), cross-fitting theo ảnh gốc, ghi dự đoán out-of-fold vào `labels/disease_clf/oof_predictions.csv`. LF2/LF3/LF4/LF5 đọc file này để bỏ phiếu correctness; LF6 lấy anchor. Notebook này **không** ghi phiếu LF.

Xem thêm tại `docs/LF3_Methodology.md`.

## Cài đặt

In [1]:
%pip install -q torchvision

## Cấu hình

In [2]:
import platform
import random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torchvision

RUNNER = 'colab'        # 'colab' | 'local' | 'kaggle'
SEED = 42
K = 5
ARCH = 'efficientnet_b0'
IMGSZ = 224
EPOCHS = 40
BATCH = 32
LR = 0.0005
PATIENCE = 6
VAL_FRAC = 0.15

CLASS_FOLDERS = ['Gray Leaf Spot', 'Leaf Rot', 'Stem Bleeding', 'Bud Rot', 'Bud Root Dropping']
CLASS_KEYS = ['gls', 'leafrot', 'stembleed', 'budrot', 'budroot']
CLASS_TASK = {}
CLASS_TASK['gls'] = '2_foliar_disease'
CLASS_TASK['leafrot'] = '2_foliar_disease'
CLASS_TASK['stembleed'] = '3_trunk_disease'
CLASS_TASK['budrot'] = '4_crown_disease'
CLASS_TASK['budroot'] = '5_petiole'
PROB_COLS = ['p_gls', 'p_leafrot', 'p_stembleed', 'p_budrot', 'p_budroot']
OOF_COLUMNS = ['image_id', 'source', 'path', 'gt_class', 'gt_task', 'pred_class', 'pred_task', 'conf', 'fold'] + PROB_COLS
SOURCE = 'coconut-tree-disease'
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cpu'
CUDA_NAME = 'none'
if torch.cuda.is_available():
    DEVICE = 'cuda'
    CUDA_NAME = torch.cuda.get_device_name(0)
elif torch.backends.mps.is_available():
    DEVICE = 'mps'

print('python:', platform.python_version())
print('torch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('numpy:', np.__version__, '| pandas:', pd.__version__)
print('device:', DEVICE, '| cuda:', CUDA_NAME)
print('runner:', RUNNER)

python: 3.12.13
torch: 2.11.0+cu128
torchvision: 0.26.0+cu128
numpy: 2.0.2 | pandas: 2.2.2
device: cuda | cuda: Tesla T4
runner: colab


## Chuẩn bị dữ liệu + module dùng chung

Colab: clone repo public lấy dataset đã commit và `src/utils/lf_io.ipynb` (`fold_of`/`original_id` — chia fold khớp LF1/LF6).

In [3]:
import subprocess

REPO_URL = 'https://github.com/phuong-duong/coconut-iqa.git'


def ensure_clone(dest):
    if dest.exists():
        return
    subprocess.run(
        ['git', 'clone', '--depth', '1', REPO_URL, str(dest)],
        check=True,
    )


def resolve_env(runner):
    if runner == 'colab':
        repo = Path('/content/coconut-iqa')
        ensure_clone(repo)
        base = repo / 'Dataset' / 'Coconut Tree Disease Dataset'
        if not base.exists():
            raise SystemExit('Khong thay ' + str(base) + ' — repo chua commit dataset?')
        return repo, base, repo / 'src' / 'utils' / 'lf_io.ipynb'
    if runner == 'local':
        for cand in [Path.cwd(), Path.cwd().parent]:
            probe = cand / 'Dataset' / 'Coconut Tree Disease Dataset'
            if probe.exists():
                return cand, probe, cand / 'src' / 'utils' / 'lf_io.ipynb'
        raise SystemExit('Khong thay Dataset/Coconut Tree Disease Dataset — chay trong repo coconut-iqa')
    if runner == 'kaggle':
        repo = Path('/kaggle/input/coconut-iqa')
        base = repo / 'Dataset' / 'Coconut Tree Disease Dataset'
        if not base.exists():
            raise SystemExit('Khong thay ' + str(base))
        return repo, base, repo / 'src' / 'utils' / 'lf_io.ipynb'
    raise SystemExit("RUNNER phai la 'colab' | 'local' | 'kaggle'")

OUT_ROOT, BASE, UTILS = resolve_env(RUNNER)
ANCHOR = BASE.parent
OUT_DIR = OUT_ROOT / 'labels' / 'disease_clf'
RUN_DIR = OUT_DIR / 'runs'
OOF_CSV = OUT_DIR / 'oof_predictions.csv'
OUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)

if not UTILS.exists():
    raise SystemExit('Khong thay ' + str(UTILS) + ' — upload src/utils/lf_io.ipynb vao /content')
get_ipython().run_line_magic('run', str(UTILS))

print('BASE:', BASE)
print('UTILS:', UTILS)
print('OOF_CSV:', OOF_CSV)

BASE: /content/coconut-iqa/Dataset/Coconut Tree Disease Dataset
UTILS: /content/coconut-iqa/src/utils/lf_io.ipynb
OOF_CSV: /content/coconut-iqa/labels/disease_clf/oof_predictions.csv


/usr/local/lib/python3.12/dist-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


## 1. Index ảnh + ground-truth (5 lớp) + gán fold theo ảnh gốc

`fold_of` và `original_id` lấy từ `src/utils/lf_io.ipynb`. Ảnh Mendeley không có augment ×3 → `original_id` = chính ảnh.

In [4]:
rows = []
for cls_folder in CLASS_FOLDERS:
    key = CLASS_KEYS[CLASS_FOLDERS.index(cls_folder)]
    cdir = BASE / cls_folder
    if not cdir.exists():
        raise SystemExit('Khong thay ' + str(cdir) + ' — kiem tra cau truc Coconut Tree Disease Dataset')
    for img in sorted(cdir.iterdir()):
        if img.name.startswith('.'):
            continue
        if img.suffix.lower() not in IMG_EXTS:
            continue
        image_id = img.stem
        oid = original_id(image_id, SOURCE)
        row = {}
        row['image_id'] = image_id
        row['source'] = SOURCE
        row['path'] = str(img.relative_to(ANCHOR))
        row['abspath'] = str(img.resolve())
        row['gt_class'] = key
        row['gt_task'] = CLASS_TASK[key]
        row['label'] = CLASS_KEYS.index(key)
        row['original_id'] = oid
        row['fold'] = fold_of(oid, K, SEED)
        rows.append(row)
df = pd.DataFrame(rows)

if len(df) == 0:
    raise SystemExit('Khong index duoc anh nao — kiem tra BASE')

spans = df.groupby('original_id').fold.nunique()
if not (spans == 1).all():
    raise SystemExit('RO RI: mot anh goc nam o nhieu fold')

print('anh:', len(df), '| anh goc:', df.original_id.nunique())
print('theo lop:', df.gt_class.value_counts().to_dict())
print('theo tac vu:', df.gt_task.value_counts().to_dict())
for k in range(K):
    sub = df[df.fold == k]
    present = sorted(sub.gt_class.unique())
    if len(present) < len(CLASS_KEYS):
        raise SystemExit('Fold ' + str(k) + ' thieu lop: ' + str(present))
    print('fold', k, '| anh', len(sub), '| theo lop', sub.gt_class.value_counts().to_dict())

anh: 5798 | anh goc: 5798
theo lop: {'gls': 2135, 'leafrot': 1673, 'stembleed': 1006, 'budroot': 514, 'budrot': 470}
theo tac vu: {'2_foliar_disease': 3808, '3_trunk_disease': 1006, '5_petiole': 514, '4_crown_disease': 470}
fold 0 | anh 1178 | theo lop {'gls': 412, 'leafrot': 334, 'stembleed': 232, 'budroot': 106, 'budrot': 94}
fold 1 | anh 1191 | theo lop {'gls': 441, 'leafrot': 353, 'stembleed': 196, 'budroot': 106, 'budrot': 95}
fold 2 | anh 1177 | theo lop {'gls': 444, 'leafrot': 327, 'stembleed': 200, 'budroot': 116, 'budrot': 90}
fold 3 | anh 1134 | theo lop {'gls': 417, 'leafrot': 340, 'stembleed': 180, 'budrot': 107, 'budroot': 90}
fold 4 | anh 1118 | theo lop {'gls': 421, 'leafrot': 319, 'stembleed': 198, 'budroot': 96, 'budrot': 84}


## 2. Classifier cross-fitting → dự đoán out-of-fold

Ảnh $x$ thuộc fold $k$ được chấm bằng model train trên $K-1$ fold còn lại (chưa từng thấy $x$) → không rò rỉ. Train class-weighted (lệch lớp 2135 vs 470). Early-stopping theo val loss trên phần tách phân tầng từ $K-1$ fold train (không đụng fold $k$).

Xem thêm tại `docs/LF3_Methodology.md`.

In [5]:
import csv
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision import models
from torchvision import transforms
from PIL import Image

NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMGSZ, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(IMGSZ + 32),
    transforms.CenterCrop(IMGSZ),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])


class DiseaseDataset(Dataset):
    def __init__(self, frame, tf):
        self.paths = list(frame.abspath)
        self.labels = list(frame.label)
        self.tf = tf

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert('RGB')
        x = self.tf(img)
        return x, self.labels[i]


def build_model():
    n_out = len(CLASS_KEYS)
    if ARCH == 'efficientnet_b0':
        from torchvision.models import EfficientNet_B0_Weights
        weights = EfficientNet_B0_Weights.IMAGENET1K_V1
        model = models.efficientnet_b0(weights=weights)
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(in_features, n_out)
        return model.to(DEVICE)
    if ARCH == 'mobilenet_v3_large':
        from torchvision.models import MobileNet_V3_Large_Weights
        weights = MobileNet_V3_Large_Weights.IMAGENET1K_V1
        model = models.mobilenet_v3_large(weights=weights)
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(in_features, n_out)
        return model.to(DEVICE)
    raise SystemExit('ARCH khong ho tro: ' + ARCH)


def class_weights(frame):
    counts = []
    for c in range(len(CLASS_KEYS)):
        counts.append(int((frame.label == c).sum()))
    total = float(sum(counts))
    out = []
    for n in counts:
        if n == 0:
            raise SystemExit('Lop rong khi tinh trong so: ' + str(counts))
        out.append(total / (len(CLASS_KEYS) * float(n)))
    return torch.tensor(out, dtype=torch.float32, device=DEVICE)


def split_train_val(frame):
    val_index = []
    for c in range(len(CLASS_KEYS)):
        sub = frame[frame.label == c]
        n_val = max(1, int(round(len(sub) * VAL_FRAC)))
        picked = sub.sample(
            n=n_val,
            random_state=SEED,
        )
        val_index.extend(picked.index.tolist())
    is_val = frame.index.isin(val_index)
    return frame[~is_val], frame[is_val]


def make_loader(frame, tf, shuffle):
    dataset = DiseaseDataset(frame, tf)
    return DataLoader(
        dataset,
        batch_size=BATCH,
        shuffle=shuffle,
        num_workers=0,
    )


def val_loss(model, loader, criterion):
    model.eval()
    total = 0.0
    n = 0
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            logits = model(images)
            loss = criterion(logits, labels)
            total = total + float(loss.item()) * len(labels)
            n = n + len(labels)
    return total / n


def train_fold(train_frame):
    inner_train, inner_val = split_train_val(train_frame)
    model = build_model()
    weights = class_weights(inner_train)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LR,
    )
    train_loader = make_loader(inner_train, train_tf, True)
    val_eval_loader = make_loader(inner_val, eval_tf, False)
    best_val = float('inf')
    best_state = None
    since_improve = 0
    for epoch in range(EPOCHS):
        model.train()
        running = 0.0
        for images, labels in train_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            running = running + float(loss.item())
        vloss = val_loss(model, val_eval_loader, criterion)
        print('  epoch', epoch, '| train_loss', round(running / len(train_loader), 4), '| val_loss', round(vloss, 4), flush=True)
        if vloss < best_val:
            best_val = vloss
            best_state = {}
            for name, tensor in model.state_dict().items():
                best_state[name] = tensor.detach().cpu().clone()
            since_improve = 0
        else:
            since_improve = since_improve + 1
        if since_improve >= PATIENCE:
            print('  early stop @ epoch', epoch, flush=True)
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def predict_fold(model, val_frame):
    model.eval()
    loader = make_loader(val_frame, eval_tf, False)
    prob_chunks = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            logits = model(images)
            probs = torch.softmax(logits, dim=1)
            prob_chunks.append(probs.cpu().numpy())
    all_probs = np.concatenate(prob_chunks)
    return all_probs

## 3. Sinh dự đoán out-of-fold → `labels/disease_clf/oof_predictions.csv`

Train cross-fitting, ghi incremental theo từng fold + resume qua checkpoint `runs/fold{k}.pt`: chạy lại bỏ qua fold đã có checkpoint và dòng oof.

In [6]:
def oof_row(rec, probs):
    best = int(np.argmax(probs))
    pred_key = CLASS_KEYS[best]
    row = {}
    row['image_id'] = rec.image_id
    row['source'] = rec.source
    row['path'] = rec.path
    row['gt_class'] = rec.gt_class
    row['gt_task'] = rec.gt_task
    row['fold'] = rec.fold
    row['pred_class'] = pred_key
    row['pred_task'] = CLASS_TASK[pred_key]
    row['conf'] = float(probs[best])
    for i in range(len(PROB_COLS)):
        row[PROB_COLS[i]] = float(probs[i])
    return row


def append_oof(path, out_rows):
    is_new = not path.exists()
    handle = path.open('a', newline='')
    writer = csv.DictWriter(handle, fieldnames=OOF_COLUMNS, extrasaction='ignore')
    if is_new:
        writer.writeheader()
    for r in out_rows:
        writer.writerow(r)
    handle.flush()
    handle.close()


done_folds = set()
if OOF_CSV.exists():
    prev = pd.read_csv(OOF_CSV)
    for k in range(K):
        ckpt = RUN_DIR / ('fold' + str(k) + '.pt')
        has_rows = bool((prev.fold == k).any())
        if ckpt.exists() and has_rows:
            done_folds.add(k)
    prev = prev[prev.fold.isin(done_folds)]
    prev.to_csv(OOF_CSV, index=False)
    print('resume: bo qua fold', sorted(done_folds), '|', len(prev), 'dong da co', flush=True)

for k in range(K):
    if k in done_folds:
        continue
    tr = df[df.fold != k]
    va = df[df.fold == k]
    print('fold', k, '| train', len(tr), '| val', len(va), flush=True)
    model = train_fold(tr)
    probs = predict_fold(model, va)
    out_rows = []
    i = 0
    for rec in va.itertuples():
        out_rows.append(oof_row(rec, probs[i]))
        i = i + 1
    append_oof(OOF_CSV, out_rows)
    torch.save(model.state_dict(), RUN_DIR / ('fold' + str(k) + '.pt'))
    print('  ghi oof + checkpoint fold', k, flush=True)
print('oof ->', OOF_CSV, flush=True)

resume: bo qua fold [] | 0 dong da co
fold 0 | train 4620 | val 1178
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 144MB/s]


  epoch 0 | train_loss 0.1533 | val_loss 0.034
  epoch 1 | train_loss 0.0415 | val_loss 0.0197
  epoch 2 | train_loss 0.0215 | val_loss 0.0181
  epoch 3 | train_loss 0.0157 | val_loss 0.0129
  epoch 4 | train_loss 0.0146 | val_loss 0.0122
  epoch 5 | train_loss 0.0202 | val_loss 0.011
  epoch 6 | train_loss 0.016 | val_loss 0.0106
  epoch 7 | train_loss 0.0071 | val_loss 0.0124
  epoch 8 | train_loss 0.0315 | val_loss 0.0389
  epoch 9 | train_loss 0.0459 | val_loss 0.0246
  epoch 10 | train_loss 0.0288 | val_loss 0.0993
  epoch 11 | train_loss 0.02 | val_loss 0.0178
  epoch 12 | train_loss 0.0097 | val_loss 0.0302
  early stop @ epoch 12
  ghi oof + checkpoint fold 0
fold 1 | train 4607 | val 1191
  epoch 0 | train_loss 0.1566 | val_loss 0.0357
  epoch 1 | train_loss 0.0383 | val_loss 0.0262
  epoch 2 | train_loss 0.0339 | val_loss 0.0268
  epoch 3 | train_loss 0.0339 | val_loss 0.0142
  epoch 4 | train_loss 0.018 | val_loss 0.0134
  epoch 5 | train_loss 0.0177 | val_loss 0.0156
  epoc

## 4. Độ chính xác out-of-fold + confusion matrix theo lớp

In [7]:
final = pd.read_csv(OOF_CSV)
has_pred = final.pred_class.notna()
has_pred = has_pred & (final.pred_class.astype(str) != '')
scored = final[has_pred].copy()
print('oof dong:', len(final), '| co pred:', len(scored))
if len(scored) == 0:
    raise SystemExit('oof rong — chay muc 3 (train cross-fitting) truoc')
else:
    class_acc = float((scored.pred_class == scored.gt_class).mean())
    task_acc = float((scored.pred_task == scored.gt_task).mean())
    print('out-of-fold accuracy (lop):', round(class_acc, 4))
    print('out-of-fold accuracy (view/tac vu):', round(task_acc, 4))
    n = len(CLASS_KEYS)
    key_index = {}
    for i in range(n):
        key_index[CLASS_KEYS[i]] = i
    cm = np.zeros((n, n), dtype=int)
    for r in scored.itertuples():
        gi = key_index[r.gt_class]
        pj = key_index[r.pred_class]
        cm[gi][pj] = cm[gi][pj] + 1
    header = 'gt\\pred'.ljust(12)
    for key in CLASS_KEYS:
        header = header + key.ljust(11)
    print(header)
    for i in range(n):
        line = CLASS_KEYS[i].ljust(12)
        for j in range(n):
            line = line + str(int(cm[i][j])).ljust(11)
        print(line)

oof dong: 5798 | co pred: 5798
out-of-fold accuracy (lop): 0.994
out-of-fold accuracy (view/tac vu): 0.9997
gt\pred     gls        leafrot    stembleed  budrot     budroot    
gls         2107       28         0          0          0          
leafrot     5          1667       0          1          0          
stembleed   0          0          1006       0          0          
budrot      1          0          0          469        0          
budroot     0          0          0          0          514        


In [8]:
import shutil
from google.colab import files

archive = shutil.make_archive('/content/disease_clf', 'zip', str(OUT_DIR))
print('zip:', archive)
files.download(archive)

zip: /content/disease_clf.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>